## 02 — Querying and Benchmarking

The index is built. Now we measure whether it is actually faster.

This notebook:
1. Verifies the grid query returns the same results as the linear scan
2. Benchmarks both methods across viewports at different zoom levels
3. Shows where the grid wins — and where it doesn't
4. Discusses what comes after a uniform grid (R-trees, quadtrees)

## Setup — Bring Both Methods Together

In [ ]:
import json
import time
from pathlib import Path

with open(Path("../../data/lod/railroads_fine.geojson")) as f:
    fine = json.load(f)

features = fine["features"]
print(f"Fine LOD: {len(features):,} features")

In [ ]:
# ── helper functions ──────────────────────────────────────────────────────────

def feature_bbox(feature):
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]


def bbox_intersects(bbox_a, bbox_b):
    a0, a1, a2, a3 = bbox_a
    b0, b1, b2, b3 = bbox_b
    if a2 < b0: return False
    if a0 > b2: return False
    if a3 < b1: return False
    if a1 > b3: return False
    return True


def linear_cull(features, viewport_bbox):
    """Original linear scan from Module 03."""
    return [f for f in features if bbox_intersects(feature_bbox(f), viewport_bbox)]


# ── grid index ────────────────────────────────────────────────────────────────

class GridIndex:
    def __init__(self, cell_size=10.0):
        self.cell_size = cell_size
        self.cells = {}
        self.n_features = 0

    def _cells_for_bbox(self, bbox):
        lon_min, lat_min, lon_max, lat_max = bbox
        col_min = int((lon_min + 180) / self.cell_size)
        col_max = int((lon_max + 180) / self.cell_size)
        row_min = int((lat_min +  90) / self.cell_size)
        row_max = int((lat_max +  90) / self.cell_size)
        return [
            (col, row)
            for col in range(col_min, col_max + 1)
            for row in range(row_min, row_max + 1)
        ]

    def build(self, features):
        self.cells = {}
        self.n_features = len(features)
        for idx, feature in enumerate(features):
            for cell in self._cells_for_bbox(feature_bbox(feature)):
                if cell not in self.cells:
                    self.cells[cell] = []
                self.cells[cell].append((idx, feature))

    def query(self, viewport_bbox):
        seen = set()
        results = []
        for cell in self._cells_for_bbox(viewport_bbox):
            for idx, feature in self.cells.get(cell, []):
                if idx not in seen:
                    seen.add(idx)
                    results.append(feature)
        return results

In [ ]:
# Build the index
index = GridIndex(cell_size=10.0)
index.build(features)
print("Index built.")

## Step 1 — Verify Correctness

Before benchmarking, confirm the grid query returns the same features as the linear scan. The sets of feature indices must match.

In [ ]:
def feature_index(feature, features):
    """Return the index of a feature in the list (by identity)."""
    return next(i for i, f in enumerate(features) if f is feature)


test_viewports = [
    ("Europe",      [-10, 35,  40, 70]),
    ("France",      [ -5, 42,  10, 52]),
    ("Paris",       [  1, 48,   4, 50]),
]

all_match = True
for name, vp in test_viewports:
    linear = set(id(f) for f in linear_cull(features, vp))
    grid   = set(id(f) for f in index.query(vp))
    match  = linear == grid
    if not match:
        all_match = False
        only_linear = linear - grid
        only_grid   = grid   - linear
        print(f"  MISMATCH {name}: only in linear={len(only_linear)}, only in grid={len(only_grid)}")
    else:
        print(f"  MATCH  {name}:  {len(linear)} features")

print()
print("All results match" if all_match else "MISMATCHES FOUND — check implementation")

## Step 2 — Benchmark

Now time both methods across five viewports representing zoom levels 2 through 13.

In [ ]:
RUNS = 100  # repeat each query this many times to get stable timings

viewports = [
    ("World z2",         [-180, -90,  180,  90]),
    ("Europe z5",        [  -10,  35,   40,  70]),
    ("France z7",        [   -5,  42,   10,  52]),
    ("Paris z10",        [    1,  48,    4,  50]),
    ("Central Paris z13",[  2.3, 48.8, 2.4, 48.9]),
]

print(f"{'Viewport':<22} {'Linear (ms)':>13} {'Grid (ms)':>11} {'Speedup':>9} {'Features':>10}")
print("-" * 70)

for name, vp in viewports:
    # Time linear scan
    t0 = time.perf_counter()
    for _ in range(RUNS):
        linear_result = linear_cull(features, vp)
    linear_ms = (time.perf_counter() - t0) / RUNS * 1000

    # Time grid query
    t0 = time.perf_counter()
    for _ in range(RUNS):
        grid_result = index.query(vp)
    grid_ms = (time.perf_counter() - t0) / RUNS * 1000

    speedup = linear_ms / grid_ms if grid_ms > 0 else float('inf')
    n = len(linear_result)

    print(f"{name:<22} {linear_ms:>12.3f} {grid_ms:>10.3f} {speedup:>8.1f}x {n:>10,}")

## Reading the Results

The speedup should increase as the viewport gets smaller:

- **World zoom** — the viewport covers all cells. The grid visits every cell and deduplicates, which adds overhead vs. a plain linear scan. The grid is **slower** here.
- **City zoom** — the viewport touches 1–2 cells with a small number of features. The grid is **much faster** than scanning 20,000 features.

This is the fundamental characteristic of spatial indexes: they help most when queries are small relative to the total dataset.

## Live Map with Grid Index

Replace the linear scan in the Module 03 live map with the grid query.

In [ ]:
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets

def leaflet_bounds_to_bbox(bounds):
    (lat_min, lon_min), (lat_max, lon_max) = bounds
    return [lon_min, lat_min, lon_max, lat_max]

m = Map(center=[48.5, 2.5], zoom=6)
status = widgets.Label(value="Pan or zoom to trigger index query")

layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"color": "#cc3300", "weight": 1.5, "opacity": 0.8}
)
m.add(layer)

def update_layer(*args):
    if not m.bounds:
        return
    vp = leaflet_bounds_to_bbox(m.bounds)

    t0 = time.perf_counter()
    visible = index.query(vp)
    elapsed_ms = (time.perf_counter() - t0) * 1000

    layer.data = {"type": "FeatureCollection", "features": visible}
    status.value = (
        f"{len(visible):,} features  |  "
        f"query: {elapsed_ms:.2f}ms  |  "
        f"viewport: {[round(v, 2) for v in vp]}"
    )

m.observe(update_layer, names=["bounds"])
update_layer()

widgets.VBox([m, status])

## Beyond the Uniform Grid

The grid index has one known weakness: **uneven data distribution**.

Railroads are not uniformly distributed across the world. Dense cells (Western Europe, eastern US) have hundreds of features. Empty cells (oceans, poles) have none.

A uniform grid cannot adapt to this — it gives every cell the same size regardless of how many features it holds.

More advanced spatial indexes solve this:

| Index | Key idea |
|-------|----------|
| **Quadtree** | Recursively subdivide cells that are too full |
| **R-tree** | Pack features into tight bounding rectangles that adapt to density |
| **KD-tree** | Binary spatial partitioning on alternating axes |

Python's `shapely` STRtree is a production R-tree. Now that you have built a grid index from scratch, you understand *why* it exists and what it improves over.

We will not build a full R-tree here — that is the industrial tool. But you know the problem it solves.

## Exercise A

The benchmark shows the grid is slower than linear scan at world zoom. Explain **why** — trace through what the grid query does when `viewport_bbox` covers all 648 cells, and compare the work to a plain list comprehension.

In [ ]:
# Write your explanation as comments
# Optionally instrument the query to count operations
# Your code here

## Exercise B

Try Shapely's `STRtree` as an alternative index. Build it from the fine LOD features and time it against the same five viewports.

```python
from shapely.strtree import STRtree
from shapely.geometry import box
```

Compare the STRtree query times to your grid index. Where does the R-tree win?

In [ ]:
# Build a Shapely STRtree from the fine LOD features
# Benchmark against the same viewports used above
# Your code here

## Check Your Understanding

Our `GridIndex` stores **references** to feature objects in multiple cells. It does not copy the features.

If a feature appears in 6 cells, how much extra memory does that use compared to storing it once? And why does deduplication in `query()` still need to happen even though we only stored references?

---

## Next

In [Module 05 — Zoom-Driven Layer Switching](../05-Zoom_Layer_Switching/README.md), we combine the LOD files with the grid index and switch both the data source and the index based on zoom level.